# 02 · Thinking in N dimensions / Pensar en N dimensiones

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb)

*Part II · demo + exercise · 20 min*

The goal of this notebook is simple:

**Do not read a tensor as a list of numbers. Read every axis as a question: “What does this axis count?”**

> 🇪🇸 El objetivo de este cuaderno es sencillo:
>
> **No leas un tensor como una lista de números. Lee cada eje como una pregunta: “¿Qué cuenta este eje?”**

## What you will be able to do / Lo que podrás hacer

- Read real image and video tensor shapes and explain every axis in plain language.
- Distinguish a **batch axis** from a **time axis**, even when the shapes are identical.
- See why shuffling independent examples can be acceptable while shuffling time changes the meaning.
- Build a padded order-5 video batch and use a validity mask to distinguish real frames from padding.

> 🇪🇸
>
> - Leer formas de tensores reales de imágenes y video y explicar cada eje en lenguaje sencillo.
> - Distinguir un **eje de lote** de un **eje temporal**, incluso cuando las formas son idénticas.
> - Entender por qué reorganizar ejemplos independientes puede ser válido mientras reorganizar el tiempo cambia el significado.
> - Construir un lote de video de orden 5 con padding y usar una máscara de validez para distinguir fotogramas reales de relleno.

## Start with an everyday example / Empecemos con un ejemplo cotidiano

Imagine two objects that both contain **8 pictures**.

### A photo album / Un álbum de fotos
The eight pictures are independent. If you change their order, you still have the same eight pictures.

### A short video / Un video corto
The eight pictures are moments in time. If you change their order, the story changes.

A computer could store both objects with the same shape:

`(8, 8, 8)`

But the first `8` can mean two completely different things:

- `N = 8` → eight independent examples / ocho ejemplos independientes
- `T = 8` → eight ordered moments / ocho momentos ordenados

**Same shape ≠ same meaning.**

> 🇪🇸 Imagina dos objetos que contienen **8 imágenes**.
>
> En un álbum, cambiar el orden de las fotos no cambia la identidad de las fotos.
>
> En un video, cambiar el orden de los fotogramas cambia la historia temporal.
>
> Ambos pueden tener la misma forma numérica, pero el significado del primer eje es diferente.

## The five axis letters we will use / Las cinco letras de ejes que usaremos

| Letter / Letra | English | Español | Simple question / Pregunta sencilla |
|---|---|---|---|
| `N` | batch / examples | lote / ejemplos | How many independent examples? / ¿Cuántos ejemplos independientes? |
| `T` | time | tiempo | How many measured moments? / ¿Cuántos momentos medidos? |
| `H` | height | alto | How many pixel rows? / ¿Cuántas filas de píxeles? |
| `W` | width | ancho | How many pixel columns? / ¿Cuántas columnas de píxeles? |
| `C` | channels | canales | How many colour or measurement channels? / ¿Cuántos canales de color o medición? |

A shape such as `(N, T, H, W, C)` is much easier to understand when you read it as a sentence:

**examples × time × height × width × channels**

> 🇪🇸 Una forma como `(N, T, H, W, C)` es más fácil de entender si la lees como una frase:
>
> **ejemplos × tiempo × alto × ancho × canales**

## Setup / Preparación

This notebook uses real data:

1. handwritten digit images,
2. a real RGB photograph,
3. a verified CC0 storm video.

The video is downloaded once and checked with SHA-256 so we know exactly which file we are using.

> 🇪🇸 Este cuaderno usa datos reales:
>
> 1. imágenes de dígitos manuscritos,
> 2. una fotografía RGB real,
> 3. un video real de tormenta CC0 verificado.
>
> El video se descarga una vez y se verifica con SHA-256 para confirmar exactamente qué archivo estamos usando.

In [ ]:
%pip install -q "imageio[ffmpeg]"

import hashlib
import io
import urllib.request

import imageio.v3 as iio
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_digits
from skimage import data

# Enable widgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

rng = np.random.default_rng(0)

# ---------------------------------------------------------------------------
# Real image data
# ---------------------------------------------------------------------------
digits = load_digits()
digit_batch = digits.images[:8].astype(np.float32)   # (N, H, W)
digit_labels = digits.target[:8]
real_digit = digit_batch[0]
real_photo = data.astronaut()                        # (H, W, C)

# ---------------------------------------------------------------------------
# Real video data: "Tormenta en l'Almadrava" by Nicolas Vigier, CC0
# ---------------------------------------------------------------------------
VIDEO_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/1/1e/"
    "Tormenta_en_l%27Almadrava.webm"
)
VIDEO_SHA256 = "e377fcdd2c79b55bce13c2c24b5dd7e412af39cd400eec548a79d0e59d79dc1b"
UA = "tensors-workshop/1.0 (https://github.com/project-delphi/tensors-workshop)"

def fetch_verified_video(url, expected_sha256, n_frames=16, stride=45):
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    raw = urllib.request.urlopen(req, timeout=120).read()

    got = hashlib.sha256(raw).hexdigest()
    if got != expected_sha256:
        raise ValueError(
            f"checksum mismatch: expected {expected_sha256}, got {got}"
        )

    frames = []
    source_indices = []

    for i, frame in enumerate(
        iio.imiter(io.BytesIO(raw), plugin="FFMPEG", extension=".webm")
    ):
        if i % stride == 0:
            frames.append(frame)
            source_indices.append(i)
            if len(frames) == n_frames:
                break

    return np.stack(frames), np.asarray(source_indices)

real_video, source_indices = fetch_verified_video(VIDEO_URL, VIDEO_SHA256)

assert real_video.shape == (16, 540, 960, 3), real_video.shape

# Same numerical shape as digit_batch: (8, 8, 8), but different semantics.
r0 = real_video.shape[1] // 2 - 4
c0 = real_video.shape[2] // 2 - 4
video_patch = (
    real_video[:8, r0:r0 + 8, c0:c0 + 8]
    .mean(axis=3)
    .astype(np.float32)
)

print("EN: Setup ready with real image and video data.")
print("ES: Preparación lista con datos reales de imágenes y video.")
print()
print("real_digit / dígito real:", real_digit.shape)
print("digit_batch / lote de dígitos:", digit_batch.shape)
print("real_photo / foto real:", real_photo.shape)
print("real_video / video real:", real_video.shape)
print("video_patch / recorte temporal:", video_patch.shape)

## Why this matters / Por qué esto importa

A `shape` tells you **how many positions exist along each axis**. It does not tell you what those positions mean.

Here we intentionally created two real tensors with the same shape:

`digit_batch.shape == (8, 8, 8)`

`video_patch.shape == (8, 8, 8)`

But:

- `digit_batch` means `(N, H, W)` → **8 independent images**
- `video_patch` means `(T, H, W)` → **8 ordered moments from a video**

The numbers are identical. The meaning is not.

### Learning habit / Hábito de aprendizaje

Before every operation:

**Predict → Run → Explain**

1. Name every axis.
2. Predict what the operation will do.
3. Run it.
4. Explain what changed in ordinary language.

> 🇪🇸 La `shape` indica cuántas posiciones existen en cada eje, pero no qué significan.
>
> Aquí `digit_batch` y `video_patch` tienen exactamente la misma forma `(8, 8, 8)`, pero uno representa **ejemplos independientes** y el otro **tiempo ordenado**.
>
> Antes de cada operación: **Predice → Ejecuta → Explica**.

## 2.1 Read real tensors as sentences / Lee tensores reales como frases

We will build the idea of “higher-order tensor” using real objects rather than empty arrays.

| Real object / Objeto real | Shape / Forma | Order / Orden | Read it as / Léelo como |
|---|---:|---:|---|
| one digit / un dígito | `(8, 8)` | 2 | height × width / alto × ancho |
| digit batch / lote de dígitos | `(8, 8, 8)` | 3 | examples × height × width / ejemplos × alto × ancho |
| RGB photo / foto RGB | `(512, 512, 3)` | 3 | height × width × colour / alto × ancho × color |
| sampled video / video muestreado | `(16, 540, 960, 3)` | 4 | time × height × width × colour / tiempo × alto × ancho × color |
| padded video batch / lote de video con padding | `(N, T, H, W, C)` | 5 | examples × time × height × width × colour / ejemplos × tiempo × alto × ancho × color |

**Important:** a higher order does not mean “better” or “more intelligent.” It simply means **more axes**.

> 🇪🇸 Un orden mayor no significa “mejor” ni “más inteligente”. Simplemente significa **más ejes**.

### Interactive axis explorer / Explorador interactivo de ejes

Choose a real object. The notebook will translate its shape into a sentence.

> 🇪🇸 Elige un objeto real. El cuaderno traducirá su forma a una frase.

In [ ]:
axis_choice = widgets.Dropdown(
    options=[
        ("One digit / Un dígito", "digit"),
        ("Digit batch / Lote de dígitos", "batch"),
        ("RGB photo / Foto RGB", "photo"),
        ("Real video / Video real", "video"),
    ],
    value="batch",
    description="Object / Objeto:",
    style={"description_width": "120px"},
)

def explain_real_tensor(choice):
    items = {
        "digit": (
            real_digit,
            "(H, W)",
            "height × width",
            "alto × ancho",
        ),
        "batch": (
            digit_batch,
            "(N, H, W)",
            "examples × height × width",
            "ejemplos × alto × ancho",
        ),
        "photo": (
            real_photo,
            "(H, W, C)",
            "height × width × colour",
            "alto × ancho × color",
        ),
        "video": (
            real_video,
            "(T, H, W, C)",
            "time × height × width × colour",
            "tiempo × alto × ancho × color",
        ),
    }

    arr, symbols, en, es = items[choice]

    print("Shape / Forma:", arr.shape)
    print("Order / Orden:", arr.ndim)
    print("Axes / Ejes:", symbols)
    print("EN:", en)
    print("ES:", es)

axis_output = widgets.interactive_output(
    explain_real_tensor,
    {"choice": axis_choice},
)

display(widgets.VBox([axis_choice, axis_output]))

## Exercise 1 — read the real shapes / Ejercicio 1 — lee las formas reales

For each real tensor:

1. print its shape and order,
2. name every axis,
3. decide whether reordering axis 0 would preserve or change the meaning.

### Helpful question / Pregunta útil

**Is axis 0 a collection of independent examples, or is it part of the internal structure of one observation?**

> 🇪🇸 Para cada tensor real:
>
> 1. imprime su forma y orden,
> 2. nombra cada eje,
> 3. decide si reorganizar el eje 0 conservaría o cambiaría el significado.
>
> Pregunta clave: **¿el eje 0 contiene ejemplos independientes o forma parte de la estructura interna de una observación?**

In [ ]:
# TODO 1 / TAREA 1
#
# Inspect / Inspecciona:
#   real_digit
#   digit_batch
#   real_photo
#   real_video
#
# EN:
# 1. Print .shape and .ndim.
# 2. Name every axis.
# 3. Explain what would happen if axis 0 were reordered.
#
# ES:
# 1. Imprime .shape y .ndim.
# 2. Nombra cada eje.
# 3. Explica qué ocurriría si se reorganizara el eje 0.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

objects = [
    (
        "one real digit / un dígito real",
        real_digit,
        "(H, W)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real digit batch / lote real de dígitos",
        digit_batch,
        "(N, H, W)",
        "axis 0 is independent examples; batch order can change",
        "el eje 0 son ejemplos independientes; el orden del lote puede cambiar",
    ),
    (
        "real RGB photo / foto RGB real",
        real_photo,
        "(H, W, C)",
        "axis 0 is image height; reordering rows scrambles the image",
        "el eje 0 es el alto; reorganizar filas desordena la imagen",
    ),
    (
        "real sampled video / video real muestreado",
        real_video,
        "(T, H, W, C)",
        "axis 0 is time; reordering it changes chronology",
        "el eje 0 es tiempo; reorganizarlo cambia la cronología",
    ),
]

for name, arr, axes, en, es in objects:
    print(name)
    print("  shape/forma:", arr.shape)
    print("  order/orden:", arr.ndim)
    print("  axes/ejes:", axes)
    print("  EN:", en)
    print("  ES:", es)
    print()

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

The number of axes gives the **order**. The dataset gives the axes their **meaning**.

A useful distinction is:

- **batch axis:** usually counts independent observations;
- **spatial axis:** position inside an image;
- **time axis:** position in an ordered sequence;
- **channel axis:** different measurements at the same position.

> 🇪🇸 El número de ejes determina el **orden**, pero el conjunto de datos determina el **significado**.
>
> - **eje de lote:** suele contar observaciones independientes;
> - **eje espacial:** indica posición dentro de una imagen;
> - **eje temporal:** indica posición dentro de una secuencia ordenada;
> - **eje de canales:** contiene diferentes mediciones en una misma posición.

</details>

## 2.2 Same shape, different meaning / Misma forma, distinto significado

Now focus on the most important comparison in this notebook:

`digit_batch.shape = (8, 8, 8)`

`video_patch.shape = (8, 8, 8)`

For the digits:

`(N, H, W)`

For the video:

`(T, H, W)`

The first axis has the same **size** (`8`) but a different **role**.

### Analogy / Analogía

Think of eight cards:

- **batch:** eight separate postcards — reorder them and you still have the same set;
- **time:** eight frames of an animation — reorder them and the motion becomes wrong.

> 🇪🇸 Piensa en ocho tarjetas:
>
> - **lote:** ocho postales independientes; cambiar el orden no cambia el conjunto;
> - **tiempo:** ocho fotogramas de una animación; cambiar el orden altera el movimiento.

### Compare the two meanings interactively / Compara los dos significados de forma interactiva

Switch between **Batch / Lote** and **Time / Tiempo**. The same numerical shape will be shown with a different interpretation.

> 🇪🇸 Cambia entre **Lote** y **Tiempo**. Verás la misma forma numérica con una interpretación diferente.

In [ ]:
meaning_toggle = widgets.ToggleButtons(
    options=[
        ("Batch / Lote", "batch"),
        ("Time / Tiempo", "time"),
    ],
    value="batch",
    description="Meaning / Significado:",
    style={"description_width": "140px"},
)

def show_same_shape_meaning(kind):
    plt.close("all")

    if kind == "batch":
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(digit_batch[i], cmap="gray")
            ax.set_title(f"N={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = independent examples / "
            "Misma forma: eje 0 = ejemplos independientes"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering N changes presentation order, not the identity of the examples.")
        print("ES: reorganizar N cambia el orden de presentación, no la identidad de los ejemplos.")

    else:
        fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
        for i, ax in enumerate(axes):
            ax.imshow(video_patch[i], cmap="gray")
            ax.set_title(f"T={i}")
            ax.axis("off")
        plt.suptitle(
            "Same shape (8,8,8): axis 0 = ordered time / "
            "Misma forma: eje 0 = tiempo ordenado"
        )
        plt.tight_layout()
        plt.show()
        print("EN: reordering T changes chronology.")
        print("ES: reorganizar T cambia la cronología.")

meaning_output = widgets.interactive_output(
    show_same_shape_meaning,
    {"kind": meaning_toggle},
)

display(widgets.VBox([meaning_toggle, meaning_output]))

## Exercise 2 — shuffle batch vs. shuffle time / Ejercicio 2 — reorganiza lote vs. tiempo

We will apply the **same permutation** to:

- the digit batch,
- the video sequence.

For the digits, images and labels must move together.

For the video, the same frames remain, but their temporal order changes.

We will also measure a simple quantity:

**mean change between consecutive sampled frames**

This is not a universal measure of “video quality.” It is just a way to show that the temporal relationships changed.

> 🇪🇸 Aplicaremos la **misma permutación** al lote de dígitos y a la secuencia de video.
>
> En los dígitos, imágenes y etiquetas deben moverse juntas.
>
> En el video, permanecen los mismos fotogramas, pero cambia el orden temporal.
>
> También mediremos el cambio promedio entre fotogramas muestreados consecutivos como evidencia de que cambió la relación temporal.

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Create perm = rng.permutation(8).
# 2. Apply it to digit_batch AND digit_labels.
# 3. Apply it to video_patch.
# 4. Print original and shuffled labels.
# 5. Compare the mean absolute change between consecutive sampled video frames.
# 6. Explain why the digit set is still the same but the video chronology is not.
#
# ES:
# 1. Crea perm = rng.permutation(8).
# 2. Aplícala a digit_batch Y digit_labels.
# 3. Aplícala a video_patch.
# 4. Imprime etiquetas originales y reorganizadas.
# 5. Compara el cambio absoluto medio entre fotogramas muestreados consecutivos.
# 6. Explica por qué el conjunto de dígitos sigue siendo el mismo pero la cronología del video no.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

perm = rng.permutation(8)

shuffled_digits = digit_batch[perm]
shuffled_labels = digit_labels[perm]
shuffled_video = video_patch[perm]

same_examples = (
    sorted(
        zip(
            digit_labels.tolist(),
            digit_batch.sum(axis=(1, 2)).round(6).tolist(),
        )
    )
    ==
    sorted(
        zip(
            shuffled_labels.tolist(),
            shuffled_digits.sum(axis=(1, 2)).round(6).tolist(),
        )
    )
)

def mean_consecutive_sampled_change(x):
    x = x.astype(np.float32)
    return float(np.mean(np.abs(x[1:] - x[:-1])))

before = mean_consecutive_sampled_change(video_patch)
after = mean_consecutive_sampled_change(shuffled_video)

print("Permutation / Permutación:", perm.tolist())
print("Original labels / Etiquetas originales:", digit_labels.tolist())
print("Shuffled labels / Etiquetas reorganizadas:", shuffled_labels.tolist())
print("Same labeled examples / Mismos ejemplos etiquetados:", same_examples)
print()

print(f"Video change before / Cambio antes: {before:.3f}")
print(f"Video change after  / Cambio después: {after:.3f}")
print(f"After/before ratio / Razón después/antes: {after / before:.2f}x")
print()

print("EN: the digit examples are the same; only their presentation order changed.")
print("ES: los ejemplos de dígitos son los mismos; solo cambió su orden de presentación.")
print("EN: the video frames are the same, but their measured chronology changed.")
print("ES: los fotogramas son los mismos, pero cambió su cronología medida.")

### See the shuffle / Observa la reorganización

Use the selector to compare:

- original digit order,
- shuffled digit order,
- original video order,
- shuffled video order.

> 🇪🇸 Usa el selector para comparar el orden original y reorganizado de los dígitos y del video.

In [ ]:
shuffle_view = widgets.Dropdown(
    options=[
        ("Digits — original / Dígitos — original", "digits_original"),
        ("Digits — shuffled / Dígitos — reorganizados", "digits_shuffled"),
        ("Video — original / Video — original", "video_original"),
        ("Video — shuffled / Video — reorganizado", "video_shuffled"),
    ],
    value="digits_original",
    description="View / Vista:",
    style={"description_width": "100px"},
)

def show_shuffle_view(view):
    plt.close("all")
    fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))

    if view == "digits_original":
        arr = digit_batch
        labels = digit_labels
        title = "Independent examples — original order / Ejemplos independientes — orden original"
        cmap = "gray"
    elif view == "digits_shuffled":
        arr = shuffled_digits
        labels = shuffled_labels
        title = "Independent examples — shuffled order / Ejemplos independientes — orden reorganizado"
        cmap = "gray"
    elif view == "video_original":
        arr = video_patch
        labels = np.arange(8)
        title = "Time sequence — original order / Secuencia temporal — orden original"
        cmap = "gray"
    else:
        arr = shuffled_video
        labels = perm
        title = "Time sequence — shuffled order / Secuencia temporal — orden reorganizado"
        cmap = "gray"

    for i, ax in enumerate(axes):
        ax.imshow(arr[i], cmap=cmap)
        ax.set_title(str(labels[i]))
        ax.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

shuffle_output = widgets.interactive_output(
    show_shuffle_view,
    {"view": shuffle_view},
)

display(widgets.VBox([shuffle_view, shuffle_output]))

<details>
<summary><strong>What did the shuffle prove? / ¿Qué demostró la reorganización?</strong></summary>

For the digit batch, the order of **independent examples** changed, but every image stayed paired with its label.

For the video, the same measured frames remained, but the chronology was altered.

**Key idea:** an operation can leave `shape` and `dtype` unchanged while still changing the scientific meaning.

> 🇪🇸 En el lote de dígitos cambió el orden de **ejemplos independientes**, pero cada imagen conservó su etiqueta.
>
> En el video permanecieron los mismos fotogramas medidos, pero se alteró la cronología.
>
> **Idea clave:** una operación puede conservar `shape` y `dtype` y aun así cambiar el significado científico.

</details>

## 2.3 Real videos can have different lengths / Los videos reales pueden tener longitudes diferentes

A machine-learning batch is usually stored as one rectangular block.

But real sequences are often different lengths:

- clip A: 4 frames,
- clip B: 7 frames,
- clip C: 5 frames.

How can one rectangular tensor contain all three?

One common answer is **padding**:

1. choose the longest length (`7`);
2. copy each real clip into a 7-frame slot;
3. fill unused positions with a placeholder value;
4. store a **mask** that tells us which positions are real.

### Analogy / Analogía

Imagine three students taking tests with 4, 7, and 5 answered questions. If the spreadsheet requires 7 columns for everyone, blank cells are **not answers**. We need a way to mark them as blank.

That is what the validity mask does.

> 🇪🇸 Imagina tres estudiantes con 4, 7 y 5 respuestas. Si una hoja de cálculo exige 7 columnas para todos, las celdas vacías **no son respuestas**.
>
> El padding crea esas posiciones necesarias para formar un bloque rectangular y la máscara indica cuáles contienen datos reales.

## Exercise 3 — build an order-5 batch / Ejercicio 3 — construye un lote de orden 5

We will take three non-overlapping measured segments from the real video:

- 4 frames,
- 7 frames,
- 5 frames.

After padding, predict:

1. the final shape,
2. which axis is `N`,
3. which axis is `T`,
4. how many `(N,T)` positions are real,
5. how many are padding.

> 🇪🇸 Tomaremos tres segmentos medidos del video real:
>
> - 4 fotogramas,
> - 7 fotogramas,
> - 5 fotogramas.
>
> Antes de ejecutar, predice la forma final, el significado de `N` y `T`, cuántas posiciones son reales y cuántas corresponden a padding.

In [ ]:
# TODO 3 / TAREA 3
#
# EN:
# 1. Spatially subsample real_video with real_video[:, ::4, ::4, :].
# 2. Build three non-overlapping clips with lengths 4, 7, and 5.
# 3. Compute T_max.
# 4. Allocate padded with shape (N, T_max, H, W, C).
# 5. Build a Boolean validity mask with shape (N, T_max).
# 6. Count measured slots and padding slots.
#
# ES:
# 1. Submuestrea espacialmente real_video con real_video[:, ::4, ::4, :].
# 2. Construye tres clips no superpuestos de longitudes 4, 7 y 5.
# 3. Calcula T_max.
# 4. Crea padded con forma (N, T_max, H, W, C).
# 5. Construye una máscara booleana de validez con forma (N, T_max).
# 6. Cuenta posiciones medidas y posiciones de padding.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

video_small = real_video[:, ::4, ::4, :]  # measured pixels, spatially subsampled

real_clips = [
    video_small[0:4],    # 4 measured frames
    video_small[4:11],   # 7 measured frames
    video_small[11:16],  # 5 measured frames
]

lengths = np.array([len(x) for x in real_clips])
T_max = int(lengths.max())
N = len(real_clips)
H, W, C = video_small.shape[1:]

padded = np.zeros((N, T_max, H, W, C), dtype=video_small.dtype)
valid = np.zeros((N, T_max), dtype=bool)

for n, x in enumerate(real_clips):
    T = len(x)
    padded[n, :T] = x
    valid[n, :T] = True

padded_slots = int((~valid).sum())
total_slots = int(valid.size)
measured_slots = int(valid.sum())

print("Clip lengths / Longitudes:", lengths.tolist())
print("Padded shape / Forma con padding:", padded.shape)
print("Order / Orden:", padded.ndim)
print("Axes / Ejes: (N, T, H, W, C)")
print("Validity mask / Máscara de validez:", valid.shape)
print("Measured frame slots / Posiciones medidas:", measured_slots)
print("Padding frame slots / Posiciones de padding:", padded_slots)
print(f"Padding fraction / Fracción de padding: {padded_slots / total_slots:.1%}")

assert padded.shape == (3, 7, 135, 240, 3)
assert valid.sum() == 16

### REAL vs PAD map / Mapa REAL vs PAD

The map below shows the batch at the `(N,T)` level.

- **REAL** = a measured video frame exists.
- **PAD** = no measured frame exists; this position only keeps the batch rectangular.

> 🇪🇸 El mapa muestra el lote en el nivel `(N,T)`.
>
> - **REAL** = existe un fotograma medido.
> - **PAD** = no existe un fotograma medido; esa posición solo mantiene rectangular el lote.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.imshow(valid, cmap="Greys", vmin=0, vmax=1, aspect="auto")

for n in range(N):
    for t in range(T_max):
        text = "REAL" if valid[n, t] else "PAD"
        text_color = "white" if valid[n, t] else "black"
        ax.text(
            t,
            n,
            text,
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
            fontweight="bold",
        )

ax.set_xticks(range(T_max))
ax.set_xlabel("Time slot T / Posición temporal T")
ax.set_yticks(range(N))
ax.set_yticklabels(
    [f"clip {n} · measured T={lengths[n]}" for n in range(N)]
)
ax.set_ylabel("Clip N / Video N")
ax.set_title(
    "Measured frames vs padding / Fotogramas medidos vs padding"
)
plt.tight_layout()
plt.show()

### Interactive padding explorer / Explorador interactivo de padding

Choose a clip and a time position.

If the slot is real, you will see the measured frame.

If the slot is padding, the notebook will show a **PAD card**, not a black image. This is important because a black image could be mistaken for a real measurement.

Try:

- `Clip N = 0, Time T = 0` → REAL
- `Clip N = 2, Time T = 6` → PAD

> 🇪🇸 Elige un clip y una posición temporal.
>
> Si la posición es real, verás el fotograma medido.
>
> Si corresponde a padding, verás una tarjeta **PAD**, no una imagen negra. Esto evita confundir un valor de relleno con una observación real.

In [ ]:
clip_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=N - 1,
    step=1,
    description="Clip N / Video N:",
    continuous_update=False,
    style={"description_width": "120px"},
)

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=T_max - 1,
    step=1,
    description="Time T / Tiempo T:",
    continuous_update=False,
    style={"description_width": "120px"},
)

def explore_padding(clip_idx, time_idx):
    slot = padded[clip_idx, time_idx]
    is_valid = bool(valid[clip_idx, time_idx])

    print(
        f"Index / Índice: padded[{clip_idx}, {time_idx}] | "
        f"shape/forma={slot.shape} | valid/válido={is_valid}"
    )

    fig, ax = plt.subplots(figsize=(6.4, 3.6))

    if is_valid:
        ax.imshow(slot)
        ax.set_title("REAL frame / Fotograma REAL")
        ax.axis("off")
        print("EN: a measured frame exists at this position.")
        print("ES: existe un fotograma medido en esta posición.")
    else:
        ax.set_facecolor("#f2f2f2")
        ax.text(
            0.5,
            0.58,
            "PAD",
            ha="center",
            va="center",
            fontsize=34,
            fontweight="bold",
            color="#993333",
            transform=ax.transAxes,
        )
        ax.text(
            0.5,
            0.38,
            "No measured frame exists here\n"
            "No existe un fotograma medido aquí",
            ha="center",
            va="center",
            fontsize=11,
            transform=ax.transAxes,
        )
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(
            "Padding only keeps the batch rectangular / "
            "El padding solo mantiene el lote rectangular"
        )
        print("EN: this slot is padding, not a measured black frame.")
        print("ES: esta posición es padding, no un fotograma negro medido.")

    plt.tight_layout()
    plt.show()

padding_output = widgets.interactive_output(
    explore_padding,
    {
        "clip_idx": clip_slider,
        "time_idx": time_slider,
    },
)

display(
    widgets.VBox([
        widgets.HBox([clip_slider, time_slider]),
        padding_output,
    ])
)

<details>
<summary><strong>Why padding needs a mask / Por qué el padding necesita una máscara</strong></summary>

The padded tensor has a convenient rectangular shape, but not every `(N,T)` position contains a measured frame.

The mask answers one simple question:

**“Should the model treat this position as real data?”**

Without the mask, the zeros used for padding could be mistaken for genuine observations.

> 🇪🇸 El tensor con padding tiene una forma rectangular conveniente, pero no toda posición `(N,T)` contiene un fotograma medido.
>
> La máscara responde una pregunta sencilla:
>
> **“¿Debe el modelo tratar esta posición como un dato real?”**
>
> Sin la máscara, los ceros usados como relleno podrían confundirse con observaciones reales.

</details>

## 2.4 The same idea in science / La misma idea en ciencia

The axis logic is not specific to videos.

Imagine a microscope experiment that repeatedly records cells.

A possible tensor could be:

`(N, T, H, W, C)`

where:

- `N` = patient, dish, well, or field of view;
- `T` = measurement time;
- `H` = image height;
- `W` = image width;
- `C` = imaging channels.

The exact convention depends on the experiment, so it must always be documented.

**Important:** a “field of view” is not automatically `H` or `W`. `H` and `W` are pixel coordinates **inside one image**.

> 🇪🇸 La misma lógica se aplica a experimentos científicos.
>
> Un tensor `(N,T,H,W,C)` podría usar:
>
> - `N` = paciente, plato, pozo o campo de visión;
> - `T` = tiempo de medición;
> - `H` = alto de la imagen;
> - `W` = ancho de la imagen;
> - `C` = canales de imagen.
>
> La convención exacta depende del experimento y debe documentarse.

### Experimental-axis explorer / Explorador de ejes experimentales

Choose an axis and read what it could mean in a microscopy experiment.

> 🇪🇸 Elige un eje y observa qué podría representar en un experimento de microscopía.

In [ ]:
experiment_axis = widgets.ToggleButtons(
    options=["N", "T", "H", "W", "C"],
    value="N",
    description="Axis / Eje:",
    style={"description_width": "90px"},
)

def explain_experiment_axis(axis):
    explanations = {
        "N": (
            "independent observation: patient, well, dish, or field of view",
            "observación independiente: paciente, pozo, plato o campo de visión",
        ),
        "T": (
            "measurement time or acquisition step",
            "tiempo de medición o paso de adquisición",
        ),
        "H": (
            "pixel rows inside one image",
            "filas de píxeles dentro de una imagen",
        ),
        "W": (
            "pixel columns inside one image",
            "columnas de píxeles dentro de una imagen",
        ),
        "C": (
            "colour, stain, fluorescence, or other measurement channels",
            "canales de color, tinción, fluorescencia u otras mediciones",
        ),
    }

    en, es = explanations[axis]
    print(f"{axis}")
    print("EN:", en)
    print("ES:", es)

experiment_output = widgets.interactive_output(
    explain_experiment_axis,
    {"axis": experiment_axis},
)

display(widgets.VBox([experiment_axis, experiment_output]))

## What just happened / Qué acaba de pasar

You used real image and video data to learn one central rule:

> **Every axis needs a meaning.**

### The four ideas to remember / Las cuatro ideas para recordar

1. **Same shape can mean different things.**  
   `(N,H,W)` and `(T,H,W)` can be numerically identical.

2. **Batch and time are not interchangeable.**  
   Reordering independent examples is different from reordering chronology.

3. **Padding is not measured data.**  
   A validity mask tells us which positions are real.

4. **Higher order just means more axes.**  
   It does not automatically mean a more complicated or better model.

### Final self-check / Autoevaluación final

If someone gives you:

`(12, 100, 64, 64, 3)`

do **not** immediately write code.

First ask:

**What do 12, 100, 64, 64, and 3 represent?**

> 🇪🇸 Si alguien te entrega un tensor `(12, 100, 64, 64, 3)`, no escribas código inmediatamente.
>
> Primero pregunta:
>
> **¿Qué representan 12, 100, 64, 64 y 3?**
>
> Esa pregunta es el comienzo de un buen razonamiento tensorial.

---

## Done with this section / Fin de esta sección

Next / Siguiente: **03 · Indexing and broadcasting real data / Indexación y broadcasting con datos reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)